# 4.3a Securing the SafeDrive ML systems

**Story:** An internal audit is coming. Until now everything ran with one broad practice role.
You give people and applications only the access they need, encrypt the data, isolate the network,
switch on auditing and scan the delivery pipeline.

**Task 4.3:** Secure ML and AI workloads and model endpoints.

| Skill | What you do here | Status |
|---|---|---|
| 4.3.3 | IAM roles and policies for a data scientist and for the claims application | ✅ |
| 4.3.2 | Least privilege on data, endpoint, model and Knowledge Base; KMS encryption | ✅ |
| 4.3.5 | Debug access problems: policy simulator, AccessDenied, CloudTrail, Access Analyzer | ✅ |
| 4.3.6 | VPC, private subnets, security group and VPC endpoints with CloudFormation | ✅ |
| 4.3.4 | CloudTrail lookup, AWS Config rules, log retention | ✅ |
| 4.3.1 | ECR image scanning, Amazon Inspector findings; CodeGuru as a note | ✅ |
| 4.3.7 | ML security risks and mitigations (GenAI risks are in 4.3b) | ✅ |

💰 The VPC interface endpoints cost about USD 0.06 per hour in total and the KMS key USD 1 per month
(pro-rated). Run the **Cleanup** section when you finish.

## Key points for the exam

- **Execution role** = assumed by a service (SageMaker, Bedrock, Lambda) to act for you.
  **User / application role** = assumed by a person or an app to call the service. Passing an
  execution role to a service needs `iam:PassRole`.
- **Least privilege** = specific actions + specific resource ARNs + conditions. An explicit **Deny**
  always wins.
- Policy types that can block a call: identity policy, resource policy (S3 bucket, KMS key), permission
  boundary, SCP, session policy, VPC endpoint policy.
- **Encryption at rest:** KMS keys on S3, training volumes (`VolumeKmsKeyId`), outputs
  (`OutputDataConfig.KmsKeyId`), endpoint storage (`KmsKeyId` on the endpoint config).
  **In transit:** TLS everywhere; inter-container traffic encryption for distributed training.
- **Network isolation:** private subnets + security groups + **VPC endpoints (PrivateLink)**;
  `EnableNetworkIsolation` stops the container from making any outbound call.
- **CloudTrail** = who called which API. **AWS Config** = is the resource configuration compliant.
  **CloudWatch Logs** = what the workload wrote.
- **Amazon Inspector** scans ECR images, EC2 and Lambda for vulnerabilities; **CodeGuru Security**
  scans source code.

| Need | Use |
|---|---|
| "Why was this call denied?" | Error message → IAM policy simulator → CloudTrail event |
| Find unused or public access | IAM Access Analyzer |
| Keep training traffic off the internet | VPC config + S3 gateway endpoint + interface endpoints |
| Block all outbound calls from a container | Network isolation |
| Prove compliance over time | AWS Config rules (+ conformance packs), CloudTrail, Audit Manager |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, os, time, datetime
import boto3, pandas as pd
from botocore.exceptions import ClientError
from common import utils as U, catchup as C
print(U.REGION, U.bucket())

Catch-up: the serverless fraud endpoint (and with it the training data and the model).

In [ ]:
ENDPOINT = C.ensure_fraud_endpoint()
state = U.load_state()
KB_ID = state.get("kb_id", "KBPLACEHOLDER")        # Knowledge Base from 3.2b, if it exists
print(ENDPOINT, "| knowledge base:", KB_ID)

Clients, names and ARNs used in the notebook.

In [ ]:
iam, sts, kms, s3 = U.client("iam"), U.client("sts"), U.client("kms"), U.client("s3")
ec2, cfn, cloudtrail = U.client("ec2"), U.client("cloudformation"), U.client("cloudtrail")
config, logs, ecr = U.client("config"), U.client("logs"), U.client("ecr")
ACCT, REGION, BUCKET = U.account_id(), U.REGION, U.bucket()
DS_ROLE, APP_ROLE = "safedrive-data-scientist", "safedrive-claims-app"
ENDPOINT_ARN = f"arn:aws:sagemaker:{REGION}:{ACCT}:endpoint/{ENDPOINT}"
KB_ARN = f"arn:aws:bedrock:{REGION}:{ACCT}:knowledge-base/{KB_ID}"
os.makedirs("scripts", exist_ok=True)              # %%writefile cells write here
pd.set_option("display.max_colwidth", 110)
print(ENDPOINT_ARN)

## 4.3.3 IAM roles and policies for users and applications

| Role type | Who assumes it | Example here |
|---|---|---|
| **Execution (service) role** | SageMaker, Bedrock, Lambda ... | `safedrive-practice-role` |
| **User role** | A person (through IAM Identity Center or federation) | `safedrive-data-scientist` |
| **Application role** | Code on Lambda, ECS, EC2 ... | `safedrive-claims-app` |

- A role has a **trust policy** (who may assume it) and **permission policies** (what it may do).
- **SageMaker Role Manager** builds least-privilege roles from personas (data scientist, MLOps,
  SageMaker compute) and ML activities, and can add VPC and KMS conditions.
- Other tools: managed policies (`AmazonSageMakerFullAccess` is broad), permission boundaries,
  tag-based access (ABAC), SageMaker domain user profiles each with their own execution role.

**📘 Concept: Caller, execution role and iam:PassRole**

Starting a SageMaker job involves **two identities**, and each needs its own permissions:

1. The **caller** (a data scientist, a pipeline, a Lambda function) calls `CreateTrainingJob`. The caller's
   policy must allow `sagemaker:CreateTrainingJob` **and** `iam:PassRole` on the execution role named in the request.
2. **SageMaker** assumes that **execution role**. This works only if the role's **trust policy** names
   `sagemaker.amazonaws.com` as principal.
3. The job then runs **with the execution role's permissions**: reading the training data in S3, pulling
   the image from ECR, writing logs and the model artifact, using the KMS key.

Why `iam:PassRole` exists: without it, anyone who may start a job could attach a powerful role to it and
let the job do things they are not allowed to do themselves (privilege escalation). So limit `PassRole` to
specific role ARNs and add the condition `iam:PassedToService`.

The same pattern applies to Bedrock (agent, Knowledge Base and batch job service roles), Lambda, Glue and
Step Functions.

Exam cue: "the user can open Studio but cannot start a training job: not authorized to perform
iam:PassRole" → fix the **caller's** policy. "The job starts, then fails with AccessDenied on S3 or KMS"
→ fix the **execution role** (or the bucket or key policy).

First look at the practice role. Its policies are broad (fine for learning, not for production).

In [ ]:
attached = iam.list_attached_role_policies(RoleName=U.ROLE_NAME)["AttachedPolicies"]
inline = iam.list_role_policies(RoleName=U.ROLE_NAME)["PolicyNames"]
print("managed:", [p["PolicyName"] for p in attached])
print("inline :", inline)
trust = iam.get_role(RoleName=U.ROLE_NAME)["Role"]["AssumeRolePolicyDocument"]
as_list = lambda x: x if isinstance(x, list) else [x]          # IAM JSON allows a string or a list
print("trusted services:", sorted(svc for st in as_list(trust["Statement"])
                                  for svc in as_list(st["Principal"].get("Service", []))))

Create the two roles. The trust policy lets identities of **this account** assume them (so that you
can test them). In production the trust would name the Identity Center role or the service
(for example `lambda.amazonaws.com`).

In [ ]:
trust_policy = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow", "Principal": {"AWS": f"arn:aws:iam::{ACCT}:root"}, "Action": "sts:AssumeRole"}]}

for name, desc in [(DS_ROLE, "SafeDrive data scientist: read processed data, run training"),
                   (APP_ROLE, "SafeDrive claims app: invoke the fraud endpoint and one FM")]:
    try:
        iam.create_role(RoleName=name, AssumeRolePolicyDocument=json.dumps(trust_policy),
                        Description=desc, MaxSessionDuration=3600, Tags=U.TAGS)
        print("created", name)
    except iam.exceptions.EntityAlreadyExistsException:
        print("exists ", name)

## 4.3.2 Least-privilege access to ML and AI artifacts

- Name the **actions** (no `sagemaker:*`), the **resources** (ARNs, prefixes) and add **conditions**.
- Useful condition keys: `s3:prefix`, `iam:PassedToService`, `sagemaker:InstanceTypes`,
  `sagemaker:VpcSubnets`, `sagemaker:VolumeKmsKey`, `aws:RequestTag/...`, `aws:ResourceTag/...`,
  `aws:SourceVpce`, `bedrock:InferenceProfileArn`.
- Artifacts to protect: training data, model artifacts in S3, model packages, endpoints, Feature
  Store, Knowledge Bases, prompts, guardrails.

**Data scientist policy, part 1: data.** Read only the processed data, list only that prefix, and an
explicit Deny on the raw data (it holds customer names).

In [ ]:
ds_data = [
    {"Sid": "ReadProcessedData", "Effect": "Allow", "Action": "s3:GetObject",
     "Resource": f"arn:aws:s3:::{BUCKET}/safedrive/processed/*"},
    {"Sid": "ListProcessedPrefix", "Effect": "Allow", "Action": "s3:ListBucket",
     "Resource": f"arn:aws:s3:::{BUCKET}",
     "Condition": {"StringLike": {"s3:prefix": ["safedrive/processed/*"]}}},
    {"Sid": "DenyRawData", "Effect": "Deny", "Action": "s3:*",
     "Resource": f"arn:aws:s3:::{BUCKET}/safedrive/raw/*"},
]
print(len(ds_data), "statements")

**Data scientist policy, part 2: training.** Create training jobs named `safedrive-*`, only on two
small instance types, and pass only the practice role and only to SageMaker.

In [ ]:
ds_training = [
    {"Sid": "RunSmallTrainingJobs", "Effect": "Allow", "Action": "sagemaker:CreateTrainingJob",
     "Resource": f"arn:aws:sagemaker:{REGION}:{ACCT}:training-job/safedrive-*",
     "Condition": {"ForAllValues:StringEquals": {"sagemaker:InstanceTypes": ["ml.m5.large", "ml.m5.xlarge"]}}},
    {"Sid": "WatchTrainingJobs", "Effect": "Allow",
     "Action": ["sagemaker:DescribeTrainingJob", "sagemaker:StopTrainingJob", "sagemaker:AddTags"],
     "Resource": f"arn:aws:sagemaker:{REGION}:{ACCT}:training-job/safedrive-*"},
    {"Sid": "PassOnlyThePracticeRole", "Effect": "Allow", "Action": "iam:PassRole",
     "Resource": U.role_arn(), "Condition": {"StringEquals": {"iam:PassedToService": "sagemaker.amazonaws.com"}}},
    {"Sid": "ReadTrainingLogs", "Effect": "Allow", "Action": ["logs:GetLogEvents", "logs:DescribeLogStreams"],
     "Resource": f"arn:aws:logs:{REGION}:{ACCT}:log-group:/aws/sagemaker/TrainingJobs:*"},
]
ds_policy = {"Version": "2012-10-17", "Statement": ds_data + ds_training}
iam.put_role_policy(RoleName=DS_ROLE, PolicyName="least-privilege", PolicyDocument=json.dumps(ds_policy))
print("policy attached to", DS_ROLE, "-", len(ds_policy["Statement"]), "statements")

**Application policy.** The claims app may invoke **one endpoint**, **one foundation model** and query
**one Knowledge Base**. Calling a model through an inference profile needs the profile ARN *and* the
foundation model ARN in each Region the profile can route to; the condition allows the model only
when it is reached through this profile.

In [ ]:
profile_arn = U.model_arn(U.TEXT_MODEL)
fm_id = U.TEXT_MODEL.split(".", 1)[1] if ":inference-profile/" in profile_arn else U.TEXT_MODEL
app_policy = {"Version": "2012-10-17", "Statement": [
    {"Sid": "InvokeFraudEndpointOnly", "Effect": "Allow", "Action": "sagemaker:InvokeEndpoint",
     "Resource": ENDPOINT_ARN},
    {"Sid": "InvokeProfile", "Effect": "Allow",
     "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": profile_arn},
    {"Sid": "InvokeModelThroughProfile", "Effect": "Allow",
     "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
     "Resource": f"arn:aws:bedrock:*::foundation-model/{fm_id}",
     "Condition": {"StringLike": {"bedrock:InferenceProfileArn": profile_arn}}},
    {"Sid": "QueryPolicyKnowledgeBase", "Effect": "Allow",
     "Action": ["bedrock:Retrieve", "bedrock:RetrieveAndGenerate"], "Resource": KB_ARN},
]}
iam.put_role_policy(RoleName=APP_ROLE, PolicyName="least-privilege", PolicyDocument=json.dumps(app_policy))
print(json.dumps(app_policy["Statement"][2], indent=1))

> 🖥️ **Console check:** *IAM → Roles → `safedrive-claims-app`*.
> On **Permissions**, open the inline policy `least-privilege` and read the JSON. On **Trust relationships**, see who may assume the role. Compare with `safedrive-practice-role`.

Compare the three roles: number of statements, wildcard actions and wildcard resources.
Fewer wildcards = smaller blast radius if the credentials leak.

In [ ]:
def summarize(name, statements):
    acts = [a for st in statements for a in as_list(st.get("Action", []))]
    res = [r for st in statements for r in as_list(st.get("Resource", []))]
    return {"role": name, "statements": len(statements), "actions": len(acts),
            "wildcard_actions": sum("*" in a for a in acts), "resource_is_star": sum(r == "*" for r in res)}

practice = [st for n in inline for st in
            as_list(iam.get_role_policy(RoleName=U.ROLE_NAME, PolicyName=n)["PolicyDocument"]["Statement"])]
pd.DataFrame([summarize(DS_ROLE, ds_policy["Statement"]), summarize(APP_ROLE, app_policy["Statement"]),
              summarize(U.ROLE_NAME + " (inline only)", practice)])

## 4.3.5 Troubleshoot and debug security issues

A fixed order saves time:
1. **Read the error message**: it names the principal, the action, the resource and often the reason
   ("no identity-based policy allows", "explicit deny in a service control policy" ...).
2. **IAM policy simulator**: test an action for a principal without calling the service.
3. **CloudTrail**: find the denied event (`errorCode`) and its full context.
4. Check the other policy layers: bucket or KMS key policy, permission boundary, SCP, VPC endpoint
   policy, session policy. For KMS-encrypted data you need `kms:Decrypt` as well.
5. **IAM Access Analyzer**: validate policies, find unused, public or cross-account access.

**📘 Concept: How IAM decides: allow, implicit deny, explicit deny**

Every request starts as **denied**. IAM then looks at all policies that apply:

1. **Any explicit `Deny`** in any policy → denied. Nothing can override it.
2. **Organization guardrails** (SCPs, RCPs), a **permissions boundary** and a **session policy**, if
   present, must each allow the action. They never grant access; they only limit it.
3. An **identity-based policy** (on the user or role) or a **resource-based policy** (S3 bucket policy,
   KMS key policy, Lambda permission, VPC endpoint policy) must allow it.
   - **Same account:** an allow in *either* one is enough.
   - **Cross-account:** *both* sides must allow.
   - **KMS is special:** the key policy must allow the access, either directly or by handing control
     to IAM policies of the account (the default key policy does that).
4. No allow found → **implicit deny**.

This is why the error message matters. "No identity-based policy allows the action" = implicit deny: add
an allow. "With an explicit deny in a ... policy" = find and change the Deny (or the SCP); adding more
allows will not help.

Exam cue: "the role has `AmazonS3FullAccess` but still gets AccessDenied on one bucket" → look for an
explicit deny in the bucket policy or an SCP, or for missing `kms:Decrypt` on the bucket's KMS key.

**Policy simulator.** A helper that asks: may this role do this action on this resource?
`allowed`, `implicitDeny` (nothing allows it) or `explicitDeny` (a Deny statement matches).

In [ ]:
def can(role, action, resource="*", context=None):
    entries = [{"ContextKeyName": k, "ContextKeyValues": v, "ContextKeyType": "stringList"}
               for k, v in (context or {}).items()]
    r = iam.simulate_principal_policy(PolicySourceArn=f"arn:aws:iam::{ACCT}:role/{role}",
                                      ActionNames=[action], ResourceArns=[resource], ContextEntries=entries)
    return r["EvaluationResults"][0]["EvalDecision"]

time.sleep(10)        # new roles and policies need a few seconds to be visible everywhere
print(can(APP_ROLE, "sagemaker:InvokeEndpoint", ENDPOINT_ARN))

Test a list of cases for both roles. Check that each decision is what you expect.

In [ ]:
s3_arn = f"arn:aws:s3:::{BUCKET}/safedrive"
cases = [
    (DS_ROLE, "s3:GetObject", f"{s3_arn}/processed/train/train.csv"),
    (DS_ROLE, "s3:GetObject", f"{s3_arn}/raw/policies.csv"),
    (DS_ROLE, "sagemaker:CreateEndpoint", "*"),
    (APP_ROLE, "sagemaker:InvokeEndpoint", ENDPOINT_ARN),
    (APP_ROLE, "sagemaker:InvokeEndpoint", ENDPOINT_ARN + "-other"),
    (APP_ROLE, "bedrock:InvokeModel", profile_arn),
    (APP_ROLE, "bedrock:InvokeModel", U.model_arn(U.LARGE_MODEL)),
    (APP_ROLE, "s3:GetObject", f"{s3_arn}/processed/train/train.csv"),
]
pd.DataFrame([{"role": r, "action": a, "resource": res.split(":")[-1][-45:], "decision": can(r, a, res)}
              for r, a, res in cases])

Conditions can be tested too: pass the context key values. The data scientist may train on
`ml.m5.large` but not on a GPU instance.

In [ ]:
job_arn = f"arn:aws:sagemaker:{REGION}:{ACCT}:training-job/safedrive-test"
for instance in ["ml.m5.large", "ml.p3.2xlarge"]:
    print(instance, "->", can(DS_ROLE, "sagemaker:CreateTrainingJob", job_arn,
                              {"sagemaker:InstanceTypes": [instance]}))

**Trigger a real AccessDenied.** Assume the application role (temporary credentials from STS) and
build a session with it.

In [ ]:
try:
    creds = sts.assume_role(RoleArn=f"arn:aws:iam::{ACCT}:role/{APP_ROLE}",
                            RoleSessionName="claims-app-test")["Credentials"]
    app = boto3.session.Session(region_name=REGION, aws_access_key_id=creds["AccessKeyId"],
                                aws_secret_access_key=creds["SecretAccessKey"],
                                aws_session_token=creds["SessionToken"])
    print(app.client("sts").get_caller_identity()["Arn"], "| expires", creds["Expiration"])
except ClientError as e:
    app = None
    print("Your identity may not assume the role (needs sts:AssumeRole):", e.response["Error"]["Message"][:150])

Call something the app role must **not** do: list all endpoints. Read the message: it names the
principal, the action, the resource and the reason.

In [ ]:
if app:
    app_sm = app.client("sagemaker")
    try:
        app_sm.list_endpoints()
    except ClientError as e:
        print("code   :", e.response["Error"]["Code"])
        print("message:", e.response["Error"]["Message"])

Now what the app role **may** do: invoke the fraud endpoint with one test claim.

In [ ]:
if app:
    row = U.read_csv(state["test_s3"].split("/", 3)[3], header=None).iloc[0, 1:]
    app_smr = app.client("sagemaker-runtime")
    r = app_smr.invoke_endpoint(EndpointName=ENDPOINT, ContentType="text/csv",
                                Body=",".join(str(v) for v in row))
    print("fraud score:", r["Body"].read().decode().strip())

The same for Bedrock: the allowed model answers, the larger model is denied.

In [ ]:
if app:
    app_brt = app.client("bedrock-runtime")
    for model in [U.TEXT_MODEL, U.LARGE_MODEL]:
        try:
            r = app_brt.converse(modelId=model, inferenceConfig={"maxTokens": 20},
                                 messages=[{"role": "user", "content": [{"text": "Say hello."}]}])
            print(model, "-> allowed:", r["output"]["message"]["content"][0]["text"][:40])
        except ClientError as e:
            print(model, "->", e.response["Error"]["Code"], "-", e.response["Error"]["Message"][:140])

**Find the denied call in CloudTrail.** Event history keeps 90 days of management events at no cost.
Events appear after about 5-15 minutes, so rerun this cell later if it is empty.

In [ ]:
def trail_events(attribute, value, minutes=120, limit=50):
    start = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)
    found = cloudtrail.lookup_events(LookupAttributes=[{"AttributeKey": attribute, "AttributeValue": value}],
                                     StartTime=start, MaxResults=limit)["Events"]
    rows = []
    for e in found:
        d = json.loads(e["CloudTrailEvent"])
        rows.append({"time": e["EventTime"], "event": e["EventName"], "source": d.get("eventSource"),
                     "who": d.get("userIdentity", {}).get("arn", "")[-50:], "error": d.get("errorCode", "")})
    return pd.DataFrame(rows, columns=["time", "event", "source", "who", "error"])

denied = trail_events("EventName", "ListEndpoints")
denied[denied["error"] != ""]

> 🖥️ **Console check:** *CloudTrail → Event history* → filter by **Event name** = `ListEndpoints` (allow 5-15 minutes).
> Open the event with the error code `AccessDenied` and read the JSON: `userIdentity` (the assumed app role), `eventSource`, `errorCode` and `errorMessage`.

**IAM Access Analyzer** policy validation: checks a policy document for errors, security warnings and
suggestions before you attach it. Try it on a careless policy.

In [ ]:
aa = U.client("accessanalyzer")
careless = {"Version": "2012-10-17", "Statement": [
    {"Effect": "Allow", "Action": ["sagemaker:*", "iam:PassRole"], "Resource": "*"}]}
findings = aa.validate_policy(policyDocument=json.dumps(careless), policyType="IDENTITY_POLICY")["findings"]
pd.DataFrame([{"type": f["findingType"], "issue": f["issueCode"], "details": f["findingDetails"][:110]} for f in findings])

Our application policy should come back with no security warnings.

In [ ]:
findings = aa.validate_policy(policyDocument=json.dumps(app_policy), policyType="IDENTITY_POLICY")["findings"]
print([(f["findingType"], f["issueCode"]) for f in findings] or "no findings")

📝 More debugging tools:
- `aws sts decode-authorization-message` decodes the encoded message some services return with a denial.
- Access Analyzer **unused access** findings and **policy generation from CloudTrail** help shrink
  a broad role to what it really uses.
- `iam:GetServiceLastAccessedDetails` (the "Last accessed" tab) shows which services a role never used.
- Typical ML cases: missing `iam:PassRole`; KMS key policy does not allow the execution role;
  VPC without an S3 endpoint (job hangs or times out, no AccessDenied); bucket policy with
  `aws:SourceVpce`; SCP that blocks a Region or a model.

## 4.3.2 (continued) Protect artifacts with encryption

- **At rest:** S3 (SSE-S3 by default; **SSE-KMS** with your own key for control and audit), EBS
  volumes of training jobs, endpoints and notebooks, Feature Store, Bedrock custom models and
  Knowledge Bases.
- **In transit:** all AWS API calls use TLS. Enable **inter-container traffic encryption** for
  distributed training (a little slower).
- A **customer managed KMS key** gives you a key policy, rotation, CloudTrail records of each use and
  the option to cut access by disabling the key.

Create (or reuse) the customer managed key `alias/safedrive` and turn on yearly rotation.

In [ ]:
try:
    KEY_ARN = kms.describe_key(KeyId="alias/safedrive")["KeyMetadata"]["Arn"]
except kms.exceptions.NotFoundException:
    KEY_ARN = kms.create_key(Description="SafeDrive ML data key",
                             Tags=[{"TagKey": t["Key"], "TagValue": t["Value"]} for t in U.TAGS])["KeyMetadata"]["Arn"]
    kms.create_alias(AliasName="alias/safedrive", TargetKeyId=KEY_ARN)
kms.enable_key_rotation(KeyId=KEY_ARN)
print(KEY_ARN, "| rotation:", kms.get_key_rotation_status(KeyId=KEY_ARN)["KeyRotationEnabled"])

Create a separate demo bucket for sensitive artifacts. (We do not change the practice bucket: after
the key is deleted in cleanup, objects encrypted with it could not be read any more.)

In [ ]:
SECURE_BUCKET = f"safedrive-secure-{ACCT}-{REGION}"
try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=SECURE_BUCKET)
    else:
        s3.create_bucket(Bucket=SECURE_BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})
except ClientError as e:
    print(e.response["Error"]["Code"])
s3.put_public_access_block(Bucket=SECURE_BUCKET, PublicAccessBlockConfiguration={
    "BlockPublicAcls": True, "IgnorePublicAcls": True, "BlockPublicPolicy": True, "RestrictPublicBuckets": True})
print(SECURE_BUCKET)

Set **default encryption** to SSE-KMS with our key. `BucketKeyEnabled` lowers the number of KMS
requests (and the KMS cost).

In [ ]:
s3.put_bucket_encryption(Bucket=SECURE_BUCKET, ServerSideEncryptionConfiguration={"Rules": [{
    "ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "aws:kms", "KMSMasterKeyID": KEY_ARN},
    "BucketKeyEnabled": True}]})
s3.get_bucket_encryption(Bucket=SECURE_BUCKET)["ServerSideEncryptionConfiguration"]["Rules"]

Add a bucket policy that **denies any request without TLS** (encryption in transit).

In [ ]:
tls_only = {"Version": "2012-10-17", "Statement": [{
    "Sid": "DenyInsecureTransport", "Effect": "Deny", "Principal": "*", "Action": "s3:*",
    "Resource": [f"arn:aws:s3:::{SECURE_BUCKET}", f"arn:aws:s3:::{SECURE_BUCKET}/*"],
    "Condition": {"Bool": {"aws:SecureTransport": "false"}}}]}
s3.put_bucket_policy(Bucket=SECURE_BUCKET, Policy=json.dumps(tls_only))
print("bucket policy set")

Upload an object without naming a key and check how it was encrypted. To read it, a principal needs
`s3:GetObject` **and** `kms:Decrypt` on the key.

In [ ]:
s3.put_object(Bucket=SECURE_BUCKET, Key="models/fraud/model-card.txt", Body=b"SafeDrive fraud model v1")
head = s3.head_object(Bucket=SECURE_BUCKET, Key="models/fraud/model-card.txt")
print(head["ServerSideEncryption"], "|", head["SSEKMSKeyId"], "| bucket key:", head.get("BucketKeyEnabled"))

> 🖥️ **Console check:** *Amazon S3 → the `safedrive-secure-...` bucket → Properties → Default encryption* (SSE-KMS, your key, bucket key on) and *Permissions → Bucket policy* (TLS only).
> Then *KMS → Customer managed keys → `safedrive`*: key policy, rotation setting and aliases.

📝 Where the KMS key goes in SageMaker and Bedrock:

| Data | API field | SDK argument |
|---|---|---|
| Training volume (EBS) | `ResourceConfig.VolumeKmsKeyId` | `volume_kms_key` |
| Model artifact and job output in S3 | `OutputDataConfig.KmsKeyId` | `output_kms_key` |
| Traffic between training containers | `EnableInterContainerTrafficEncryption` | `encrypt_inter_container_traffic` |
| Endpoint storage volume | `KmsKeyId` on `CreateEndpointConfig` | `kms_key` in `deploy()` |
| Data capture, batch transform output, async output | `KmsKeyId` in the respective config | |
| Processing job, notebook instance, Studio domain (EFS) | `VolumeKmsKeyId` / `KmsKeyId` | |
| Bedrock custom model, Knowledge Base data, guardrail, agent | customer managed key at creation | |

IAM can force it: deny `sagemaker:CreateTrainingJob` unless `sagemaker:VolumeKmsKey` and
`sagemaker:OutputKmsKey` are set.

## 4.3.6 Isolate ML systems with a VPC, subnets and security groups

- **Private subnets** (no internet gateway, no NAT) in 2 AZs.
- **Security group**: stateful firewall on the network interfaces of jobs and endpoints.
- **VPC endpoints** keep AWS traffic on the AWS network: **gateway** endpoint for S3 (free),
  **interface** endpoints (PrivateLink) for SageMaker API, SageMaker Runtime, Bedrock Runtime ... (paid).
- `VpcConfig` on a training job or model puts its network interfaces in your subnets.
  `EnableNetworkIsolation` goes further: the container has **no** network access at all (SageMaker
  still copies data from and to S3 for it).

**📘 Concept: VpcConfig, network isolation and VPC endpoints are three different things**

| Setting | What it does | What it does not do |
|---|---|---|
| **`VpcConfig`** (subnets + security groups) on a training job, processing job or model | The container's network traffic goes through network interfaces in **your VPC**. Your route tables, security groups and endpoint policies apply. | It does not block the internet by itself: with a NAT gateway the container can still call out. |
| **`EnableNetworkIsolation`** | The container can make **no outbound network calls at all**, not even to S3 or other AWS services. SageMaker itself copies the input data in and the model artifact out. | It does not put anything in your VPC. Code that downloads packages or calls an API at run time fails. |
| **VPC endpoints** (gateway for S3; interface endpoints for other services) | Give resources in private subnets a **private path to AWS services** without internet gateway or NAT. | They are not a setting of the job; they are part of the VPC the job runs in. |

How they combine:
- `VpcConfig` + private subnets + an **S3 gateway endpoint** = traffic stays on the AWS network and the job can still read its data.
- Add `EnableNetworkIsolation` for the strictest setup (sensitive data, untrusted code): the container is cut off completely.

Exam cue: "the training job in a private subnet cannot download its data / times out" → missing S3 VPC
endpoint (or NAT). "Ensure the training container cannot send data anywhere" → network isolation.
"Traffic to the SageMaker API or runtime must not cross the public internet" → interface VPC endpoints (PrivateLink).

Not every AZ offers every interface endpoint. Ask EC2 which AZs support all three services and pick two.

In [ ]:
services = [f"com.amazonaws.{REGION}.{s}" for s in ["sagemaker.api", "sagemaker.runtime", "bedrock-runtime"]]
details = ec2.describe_vpc_endpoint_services(ServiceNames=services)["ServiceDetails"]
common_azs = sorted(set.intersection(*[set(d["AvailabilityZones"]) for d in details]))
AZ1, AZ2 = common_azs[:2]
print("AZs with all three endpoint services:", common_azs, "-> using", AZ1, AZ2)

The CloudFormation template, part 1: the VPC, two private subnets and a route table **without** a
route to the internet.

In [ ]:
%%writefile scripts/vpc_isolation.yaml
AWSTemplateFormatVersion: "2010-09-09"
Description: SafeDrive isolated VPC for ML workloads (private subnets, VPC endpoints, no internet)
Parameters:
  Az1: {Type: "AWS::EC2::AvailabilityZone::Name"}
  Az2: {Type: "AWS::EC2::AvailabilityZone::Name"}
Resources:
  Vpc:
    Type: AWS::EC2::VPC
    Properties: {CidrBlock: 10.20.0.0/16, EnableDnsSupport: true, EnableDnsHostnames: true,
                 Tags: [{Key: Name, Value: safedrive-ml-vpc}]}
  SubnetA:
    Type: AWS::EC2::Subnet
    Properties: {VpcId: !Ref Vpc, CidrBlock: 10.20.1.0/24, AvailabilityZone: !Ref Az1,
                 Tags: [{Key: Name, Value: safedrive-private-a}]}
  SubnetB:
    Type: AWS::EC2::Subnet
    Properties: {VpcId: !Ref Vpc, CidrBlock: 10.20.2.0/24, AvailabilityZone: !Ref Az2,
                 Tags: [{Key: Name, Value: safedrive-private-b}]}
  RouteTable:
    Type: AWS::EC2::RouteTable
    Properties: {VpcId: !Ref Vpc}
  AssocA:
    Type: AWS::EC2::SubnetRouteTableAssociation
    Properties: {SubnetId: !Ref SubnetA, RouteTableId: !Ref RouteTable}
  AssocB:
    Type: AWS::EC2::SubnetRouteTableAssociation
    Properties: {SubnetId: !Ref SubnetB, RouteTableId: !Ref RouteTable}

Part 2: the security group (HTTPS only; inbound only from inside the VPC) and the S3 **gateway**
endpoint. Outbound 443 is needed to reach S3 through the gateway endpoint; there is still no route
to the internet.

In [ ]:
%%writefile -a scripts/vpc_isolation.yaml

  SecurityGroup:
    Type: AWS::EC2::SecurityGroup
    Properties:
      GroupDescription: SafeDrive ML workloads - HTTPS only
      VpcId: !Ref Vpc
      SecurityGroupIngress: [{IpProtocol: tcp, FromPort: 443, ToPort: 443, CidrIp: 10.20.0.0/16}]
      SecurityGroupEgress: [{IpProtocol: tcp, FromPort: 443, ToPort: 443, CidrIp: 0.0.0.0/0}]
  S3Endpoint:
    Type: AWS::EC2::VPCEndpoint
    Properties:
      VpcId: !Ref Vpc
      VpcEndpointType: Gateway
      ServiceName: !Sub "com.amazonaws.${AWS::Region}.s3"
      RouteTableIds: [!Ref RouteTable]

Part 3: three **interface** endpoints with private DNS (the normal service host names then resolve
to private IP addresses), and the outputs.

In [ ]:
%%writefile -a scripts/vpc_isolation.yaml

  SageMakerApiEndpoint:
    Type: AWS::EC2::VPCEndpoint
    Properties: {VpcId: !Ref Vpc, VpcEndpointType: Interface, PrivateDnsEnabled: true,
                 SubnetIds: [!Ref SubnetA, !Ref SubnetB], SecurityGroupIds: [!Ref SecurityGroup],
                 ServiceName: !Sub "com.amazonaws.${AWS::Region}.sagemaker.api"}
  SageMakerRuntimeEndpoint:
    Type: AWS::EC2::VPCEndpoint
    Properties: {VpcId: !Ref Vpc, VpcEndpointType: Interface, PrivateDnsEnabled: true,
                 SubnetIds: [!Ref SubnetA, !Ref SubnetB], SecurityGroupIds: [!Ref SecurityGroup],
                 ServiceName: !Sub "com.amazonaws.${AWS::Region}.sagemaker.runtime"}
  BedrockRuntimeEndpoint:
    Type: AWS::EC2::VPCEndpoint
    Properties: {VpcId: !Ref Vpc, VpcEndpointType: Interface, PrivateDnsEnabled: true,
                 SubnetIds: [!Ref SubnetA, !Ref SubnetB], SecurityGroupIds: [!Ref SecurityGroup],
                 ServiceName: !Sub "com.amazonaws.${AWS::Region}.bedrock-runtime"}
Outputs:
  VpcId: {Value: !Ref Vpc}
  SubnetIds: {Value: !Join [",", [!Ref SubnetA, !Ref SubnetB]]}
  SecurityGroupId: {Value: !Ref SecurityGroup}

Create the stack (about 3-4 minutes; the interface endpoints take the longest). 💰 Billing for the
interface endpoints starts now.

In [ ]:
STACK = "safedrive-ml-vpc"
try:
    cfn.describe_stacks(StackName=STACK)
    print("stack exists")
except ClientError:
    cfn.create_stack(StackName=STACK, TemplateBody=open("scripts/vpc_isolation.yaml").read(), Tags=U.TAGS,
                     Parameters=[{"ParameterKey": "Az1", "ParameterValue": AZ1},
                                 {"ParameterKey": "Az2", "ParameterValue": AZ2}])
    print("creating ...")
cfn.get_waiter("stack_create_complete").wait(StackName=STACK)
print(cfn.describe_stacks(StackName=STACK)["Stacks"][0]["StackStatus"])

Read the stack outputs: the IDs you pass to SageMaker.

In [ ]:
outputs = {o["OutputKey"]: o["OutputValue"] for o in cfn.describe_stacks(StackName=STACK)["Stacks"][0]["Outputs"]}
VPC_ID, SUBNETS, SG_ID = outputs["VpcId"], outputs["SubnetIds"].split(","), outputs["SecurityGroupId"]
print(VPC_ID, SUBNETS, SG_ID)

Check the isolation: the route table has only the local route and the S3 prefix-list route of the
gateway endpoint. There is no `0.0.0.0/0` route, so nothing can reach the internet.

In [ ]:
tables = ec2.describe_route_tables(Filters=[{"Name": "vpc-id", "Values": [VPC_ID]}])["RouteTables"]
routes = [{"destination": r.get("DestinationCidrBlock") or r.get("DestinationPrefixListId"),
           "target": r.get("GatewayId") or r.get("NatGatewayId")} for t in tables for r in t["Routes"]]
pd.DataFrame(routes).drop_duplicates()

List the VPC endpoints: one gateway endpoint (S3) and three interface endpoints with private DNS.

In [ ]:
vpces = ec2.describe_vpc_endpoints(Filters=[{"Name": "vpc-id", "Values": [VPC_ID]}])["VpcEndpoints"]
pd.DataFrame([{"service": v["ServiceName"], "type": v["VpcEndpointType"], "state": v["State"],
               "private_dns": v.get("PrivateDnsEnabled")} for v in vpces])

> 🖥️ **Console check:** *CloudFormation → Stacks → `safedrive-ml-vpc`*: the **Resources** tab lists everything the template created, **Events** shows the order.
> Then *VPC → Your VPCs → `safedrive-ml-vpc`* → **Resource map**: two subnets, one route table, no internet gateway. *VPC → Endpoints* lists the four endpoints.

Configure a training job for this VPC with every security switch on. Creating the `Estimator` object
costs nothing; look at the settings that will go into `CreateTrainingJob`.

In [ ]:
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

sm_session = sagemaker.Session(boto_session=U.session, default_bucket=BUCKET)
secure_est = Estimator(
    image_uri=state["xgb_image"], role=U.role_arn(), instance_count=1, instance_type="ml.m5.large",
    output_path=U.s3_uri("safedrive/models-secure"), sagemaker_session=sm_session,
    base_job_name="safedrive-fraud-secure", tags=U.TAGS,
    subnets=SUBNETS, security_group_ids=[SG_ID],        # VpcConfig
    enable_network_isolation=True,                      # EnableNetworkIsolation
    encrypt_inter_container_traffic=True,               # EnableInterContainerTrafficEncryption
    volume_kms_key=KEY_ARN, output_kms_key=KEY_ARN)     # VolumeKmsKeyId, OutputDataConfig.KmsKeyId
secure_est.set_hyperparameters(objective="binary:logistic", num_round=50, max_depth=4)
print({"subnets": secure_est.subnets, "security_groups": secure_est.security_group_ids,
       "network_isolation": secure_est.enable_network_isolation(), "volume_kms_key": secure_est.volume_kms_key})

🔒 Run the training job inside the VPC (about 4 minutes, a few cents). It works without internet
because SageMaker reaches S3 through the gateway endpoint.

In [ ]:
if U.expensive("training job in the isolated VPC"):
    secure_est.fit({"train": TrainingInput(state["train_s3"], content_type="text/csv"),
                    "validation": TrainingInput(state["validation_s3"], content_type="text/csv")}, logs=False)
    job = U.client("sagemaker").describe_training_job(TrainingJobName=secure_est.latest_training_job.name)
    print(job["TrainingJobStatus"], "| VpcConfig:", job["VpcConfig"], "| isolation:", job["EnableNetworkIsolation"])

📝 More isolation points:
- **Endpoints / models:** `VpcConfig` on `CreateModel`; callers reach the endpoint through the
  `sagemaker.runtime` interface endpoint. Add an **endpoint policy** to limit which endpoints may be invoked.
- **Notebooks / Studio:** `DirectInternetAccess=Disabled`, Studio domain in `VpcOnly` mode.
- **Bedrock:** interface endpoints `bedrock`, `bedrock-runtime`, `bedrock-agent`,
  `bedrock-agent-runtime`; restrict IAM with `aws:SourceVpce` / `aws:SourceVpc`.
- **Enforce with IAM:** deny `sagemaker:Create*` unless `sagemaker:VpcSubnets` and
  `sagemaker:VpcSecurityGroupIds` are set.
- Security group = stateful, allow rules only, on the network interface. Network ACL = stateless,
  allow and deny rules, on the subnet.

## 4.3.4 Monitoring, auditing, compliance and logging

| Service | Records | Typical question |
|---|---|---|
| **CloudTrail** | API calls (management events; data events if enabled) | Who deleted the endpoint? Who changed the guardrail? |
| **AWS Config** | Resource configuration over time, rule compliance | Are all endpoint configs encrypted with KMS? |
| **CloudWatch Logs** | Output of jobs, endpoints, Lambda; Bedrock invocation logs | What did the job print? What was the prompt? |
| SageMaker lineage, Model Cards, Model Registry | Which data and code produced which model; approvals | Can we reproduce and explain this model? |

- CloudTrail **event history**: 90 days, management events only. A **trail** or CloudTrail Lake keeps
  events longer (S3) and can add **data events**.
- SageMaker `InvokeEndpoint` and Bedrock agent / Knowledge Base / guardrail runtime calls are **data
  events** (off by default). Bedrock `InvokeModel` and `Converse` are management events.

Who did what in SageMaker in the last 2 hours? (The helper `trail_events` is from section 4.3.5.)

In [ ]:
sm_events = trail_events("EventSource", "sagemaker.amazonaws.com")
print(len(sm_events), "events")
sm_events.groupby(["event", "error"]).size().sort_values(ascending=False).head(10)

The same for Bedrock: model invocations and configuration changes, with the calling identity.
The denied `Converse` call of the app role shows up here with `error = AccessDenied`.

In [ ]:
br_events = trail_events("EventSource", "bedrock.amazonaws.com")
br_events.head(8)

Is there a trail that keeps events longer than 90 days? An empty list means only event history is on.

In [ ]:
trails = cloudtrail.describe_trails()["trailList"]
print([{"name": t["Name"], "bucket": t.get("S3BucketName"), "multi_region": t.get("IsMultiRegionTrail")}
       for t in trails] or "no trail - only the 90-day event history")

📝 Enable data events for endpoint invocations on an existing trail (not run here; data events are billed):

```python
cloudtrail.put_event_selectors(TrailName="my-trail", AdvancedEventSelectors=[{
    "Name": "SageMaker endpoint invocations",
    "FieldSelectors": [{"Field": "eventCategory", "Equals": ["Data"]},
                       {"Field": "resources.type", "Equals": ["AWS::SageMaker::Endpoint"]}]}])
```

**AWS Config.** Managed rules check resources continuously. They need a **configuration recorder**
in the account. Check whether one exists.

In [ ]:
recorders = config.describe_configuration_recorders()["ConfigurationRecorders"]
HAS_CONFIG = len(recorders) > 0
print("configuration recorder:", [r["name"] for r in recorders] or "none (AWS Config is not set up in this Region)")

Add two managed rules for SageMaker (only if a recorder exists; each rule evaluation costs a
fraction of a cent).

In [ ]:
RULES = {"safedrive-sagemaker-notebook-no-internet": "SAGEMAKER_NOTEBOOK_NO_DIRECT_INTERNET_ACCESS",
         "safedrive-sagemaker-endpoint-config-kms": "SAGEMAKER_ENDPOINT_CONFIGURATION_KMS_KEY_CONFIGURED"}
if HAS_CONFIG:
    for name, identifier in RULES.items():
        config.put_config_rule(ConfigRule={"ConfigRuleName": name,
                                           "Source": {"Owner": "AWS", "SourceIdentifier": identifier}})
        print("rule created:", name)
else:
    print("No recorder: rules not created. To try it, set up AWS Config (1-click setup in the console) and rerun.")

Read the compliance result (evaluation takes a few minutes; rerun if it says `INSUFFICIENT_DATA`).
Our practice endpoint configs have no KMS key, so that rule should report `NON_COMPLIANT`.

In [ ]:
if HAS_CONFIG:
    result = config.describe_compliance_by_config_rule(ConfigRuleNames=list(RULES))["ComplianceByConfigRules"]
    display(pd.DataFrame([{"rule": r["ConfigRuleName"], "compliance": r["Compliance"]["ComplianceType"]} for r in result]))
else:
    print("skipped (no configuration recorder)")

> 🖥️ **Console check:** If your account has a configuration recorder: *AWS Config → Rules → `safedrive-sagemaker-endpoint-config-kms`*.
> The rule page lists the resources in scope, each marked compliant or noncompliant; open one to see its configuration timeline.

📝 Other SageMaker managed rules: `sagemaker-notebook-instance-kms-key-configured`,
`sagemaker-notebook-instance-root-access-check`, `sagemaker-endpoint-config-prod-instance-count`,
`sagemaker-model-in-vpc`. **Conformance packs** bundle rules; **Security Hub** and **Audit Manager**
collect the findings as evidence.

**Log retention.** Log groups keep data forever by default. Set a retention period on the endpoint
log group (cost and data-minimisation). Log groups can also be encrypted with a KMS key.

In [ ]:
group = f"/aws/sagemaker/Endpoints/{ENDPOINT}"
try:
    logs.put_retention_policy(logGroupName=group, retentionInDays=90)
    g = logs.describe_log_groups(logGroupNamePrefix=group)["logGroups"][0]
    print(g["logGroupName"], "| retention days:", g.get("retentionInDays"), "| kms:", g.get("kmsKeyId", "default encryption"))
except ClientError as e:
    print("log group not found yet (invoke the endpoint first):", e.response["Error"]["Code"])

## 4.3.1 Secure CI/CD: scan code and images

- **Amazon ECR** scanning: *basic* (scan on push, OS packages) or *enhanced* (**Amazon Inspector**:
  OS and language packages, continuous re-scan when new CVEs appear).
- **Amazon Inspector** also scans Lambda functions and EC2, and offers a CI/CD scan (SBOM) for images
  before they are pushed.
- **Amazon CodeGuru Security** scans source code (Python, Java ...) for vulnerabilities and
  hard-coded secrets from a pipeline step or the IDE. **CodeGuru Reviewer** comments on pull requests
  (check current availability for new accounts).
- In the pipeline: add a **scan stage that fails the build** on HIGH or CRITICAL findings, use
  immutable image tags, sign images, and give CodeBuild and CodePipeline least-privilege roles.

Create an ECR repository for the SafeDrive inference image with **scan on push**, immutable tags and
KMS encryption at rest.

In [ ]:
REPO = "safedrive-inference"
try:
    ecr.create_repository(repositoryName=REPO, imageTagMutability="IMMUTABLE", tags=U.TAGS,
                          imageScanningConfiguration={"scanOnPush": True},
                          encryptionConfiguration={"encryptionType": "KMS"})
except ecr.exceptions.RepositoryAlreadyExistsException:
    pass
repo = ecr.describe_repositories(repositoryNames=[REPO])["repositories"][0]
print(repo["repositoryUri"], "|", repo["imageScanningConfiguration"], "|", repo["imageTagMutability"])

Check the registry scanning mode: `BASIC` or `ENHANCED` (Inspector).

In [ ]:
scan_cfg = ecr.get_registry_scanning_configuration()["scanningConfiguration"]
print("scan type:", scan_cfg["scanType"], "| rules:", scan_cfg.get("rules", []))

🔒 Switch the registry to **enhanced scanning** for `safedrive-*` repositories. This activates
Amazon Inspector for ECR in the account (billed per image scanned; 15-day free trial for new accounts).

In [ ]:
if U.expensive("ECR enhanced scanning with Amazon Inspector"):
    ecr.put_registry_scanning_configuration(scanType="ENHANCED", rules=[
        {"scanFrequency": "CONTINUOUS_SCAN",
         "repositoryFilters": [{"filter": "safedrive-*", "filterType": "WILDCARD"}]}])
    print(ecr.get_registry_scanning_configuration()["scanningConfiguration"])

Query **Amazon Inspector** for HIGH and CRITICAL findings on this repository. Without images or
with Inspector switched off the result is empty (or an access error, which the cell prints).

In [ ]:
inspector = U.client("inspector2")
try:
    found = inspector.list_findings(maxResults=20, filterCriteria={
        "ecrImageRepositoryName": [{"comparison": "EQUALS", "value": REPO}],
        "severity": [{"comparison": "EQUALS", "value": "CRITICAL"}, {"comparison": "EQUALS", "value": "HIGH"}],
        "findingStatus": [{"comparison": "EQUALS", "value": "ACTIVE"}]})["findings"]
    print(len(found), "active HIGH/CRITICAL findings")
    display(pd.DataFrame([{"severity": f["severity"], "title": f["title"], "type": f["type"]} for f in found]))
except ClientError as e:
    print("Inspector not enabled or no access:", e.response["Error"]["Code"])

📝 The pipeline gate. After `docker push`, a CodeBuild step waits for the scan and fails on serious
findings (basic scanning shown; with enhanced scanning query Inspector as above):

```yaml
# buildspec.yml (post_build)
- aws ecr wait image-scan-complete --repository-name safedrive-inference --image-id imageTag=$TAG
- CRIT=$(aws ecr describe-image-scan-findings --repository-name safedrive-inference --image-id imageTag=$TAG
         --query "imageScanFindings.findingSeverityCounts.CRITICAL" --output text)
- if [ "$CRIT" != "None" ]; then echo "critical vulnerabilities found"; exit 1; fi
```

Code scanning step with CodeGuru Security: zip the source, `create-upload-url`, upload,
`create-scan`, then `get-findings` and fail the build on `Critical` or `High`.

## 4.3.7 Security risks in ML systems and how to mitigate them

📝

| Risk | What happens | Mitigation on AWS |
|---|---|---|
| **Data poisoning** | Attacker changes training data or labels | Least-privilege S3 access, versioning, data validation and quality checks, lineage, Model Monitor |
| **Model theft / extraction** | Artifact copied, or model rebuilt from many queries | KMS-encrypted artifacts, tight IAM on S3 and registry, private endpoints, throttling, CloudTrail |
| **Adversarial input / evasion** | Crafted input causes a wrong prediction | Input validation, robust training, anomaly detection on inputs, human review for high-risk cases |
| **Model inversion / membership inference** | Training data leaks through predictions | Remove PII before training, limit output detail, differential privacy |
| **Exposed endpoints** | Endpoint or notebook reachable by anyone with credentials or from the internet | IAM-authenticated endpoints, VPC + PrivateLink, endpoint policies, API Gateway + WAF in front |
| **Over-permissive roles** | Leaked credentials can do everything | Least privilege, Role Manager, permission boundaries, Access Analyzer |
| **Secrets in notebooks or code** | API keys and passwords in Git or notebooks | **AWS Secrets Manager** / Parameter Store, IAM roles instead of keys, CodeGuru Security secret detection |
| **Vulnerable containers and packages** | Known CVEs, malicious dependencies, unsafe pickle files | ECR scanning, Inspector, trusted base images, private package repository (CodeArtifact) |
| **Unencrypted or public data** | Data read by the wrong people | S3 Block Public Access, SSE-KMS, TLS-only bucket policy, Macie to find PII |

**Secrets Manager** instead of secrets in the notebook: store a (fake) third-party API key and read
it back at run time. Access is controlled by IAM and every read is in CloudTrail.

In [ ]:
secrets = U.client("secretsmanager")
SECRET = "safedrive/demo/vehicle-data-api-key"
try:
    secrets.create_secret(Name=SECRET, SecretString=json.dumps({"api_key": "demo-not-a-real-key"}),
                          Description="Demo secret for notebook 4.3a", Tags=U.TAGS)
except secrets.exceptions.ResourceExistsException:
    pass
value = json.loads(secrets.get_secret_value(SecretId=SECRET)["SecretString"])
print("read at run time:", {k: v[:4] + "..." for k, v in value.items()})

## Exam traps

- `EnableNetworkIsolation` (no outbound calls from the container) is **not** the same as running in a
  VPC (`VpcConfig`). They can be combined.
- A training job in a private subnet **without an S3 endpoint or NAT** cannot read its data: it
  fails or hangs, and it is not an IAM problem.
- S3 uses a **gateway** endpoint (free, route table entry); most other services use **interface**
  endpoints (PrivateLink, hourly cost, security group).
- CloudTrail event history does **not** include data events such as `InvokeEndpoint`.
- With SSE-KMS a principal needs permissions on **both** the S3 object and the KMS key.

## Quiz

**1.** A data scientist starts a training job and gets `AccessDenied ... is not authorized to perform:
iam:PassRole`. What is missing?
A) The execution role lacks S3 access  B) The data scientist's policy must allow `iam:PassRole` on the execution role
C) The KMS key policy  D) A VPC endpoint

<details><summary>Answer</summary>B. The caller needs iam:PassRole to hand the execution role to SageMaker.</details>

**2.** Training data is sensitive. The training container must not be able to send data anywhere,
even if the code is malicious. Which setting?
A) Inter-container traffic encryption  B) A public subnet with a security group
C) `EnableNetworkIsolation=True`  D) SSE-KMS on the output bucket

<details><summary>Answer</summary>C. Network isolation removes all outbound network access from the container.</details>

**3.** An auditor asks who changed the endpoint configuration last Tuesday. Where do you look?
A) CloudWatch metrics  B) CloudTrail  C) Model Monitor reports  D) VPC Flow Logs

<details><summary>Answer</summary>B. CloudTrail records the UpdateEndpoint / CreateEndpointConfig API calls with the identity.</details>

**4.** A company must continuously verify that no SageMaker notebook instance has direct internet access.
A) AWS Config managed rule `sagemaker-notebook-no-direct-internet-access`  B) An Inspector scan
C) A CloudWatch alarm on CPU  D) A Macie job

<details><summary>Answer</summary>A. AWS Config evaluates resource configuration against rules continuously.</details>

**5.** The CI/CD pipeline builds a custom inference image. Images with critical CVEs must never be deployed.
A) Enable ECR scan on push / Inspector enhanced scanning and fail the pipeline on critical findings
B) Encrypt the repository with KMS  C) Use a larger build instance  D) Enable CloudTrail data events

<details><summary>Answer</summary>A. Image scanning finds the CVEs; the pipeline gate stops the deployment.</details>

## Save state

Nothing here is needed by later notebooks; all resources are deleted below.

In [ ]:
U.save_state(security_notebook_done=True)
print("saved")

## Cleanup

Everything this notebook created is deleted. Kept: the serverless fraud endpoint (no idle cost) and
the 90-day retention on its log group.

### Before you delete: what later notebooks use

The cleanup cells below **do not delete** the items in this table.

| Kept | Used by later notebooks | If you delete it anyway |
|---|---|---|
| Knowledge Base, guardrail, policy chunks (from earlier notebooks) | 4.3b | Rebuilt by the catch-up cell |
| Serverless endpoint `safedrive-fraud` | Not used again | - |

- **Continuing to the next notebook soon?** Run the cleanup cells and keep the items above. None of them bills while idle (only a little storage).
- **Stopping for good?** Run `99_cleanup/99_teardown.ipynb`; it removes everything, including these.

Delete the VPC stack (stops the interface endpoint cost) and wait until it is gone.

In [ ]:
U.safe(cfn.delete_stack, StackName="safedrive-ml-vpc")
U.safe(cfn.get_waiter("stack_delete_complete").wait, StackName="safedrive-ml-vpc")

Empty and delete the demo bucket.

In [ ]:
SECURE_BUCKET = f"safedrive-secure-{U.account_id()}-{U.REGION}"
if U.safe(s3.head_bucket, Bucket=SECURE_BUCKET):
    for obj in s3.list_objects_v2(Bucket=SECURE_BUCKET).get("Contents", []):
        s3.delete_object(Bucket=SECURE_BUCKET, Key=obj["Key"])
    U.safe(s3.delete_bucket, Bucket=SECURE_BUCKET)

Remove the alias and schedule the KMS key for deletion (7 days is the minimum waiting period;
a key pending deletion is not billed).

In [ ]:
key_id = U.safe(kms.describe_key, KeyId="alias/safedrive")
U.safe(kms.delete_alias, AliasName="alias/safedrive")
if key_id:
    U.safe(kms.schedule_key_deletion, KeyId=key_id["KeyMetadata"]["KeyId"], PendingWindowInDays=7)

Delete the two IAM roles (inline policy first).

In [ ]:
for role in ["safedrive-data-scientist", "safedrive-claims-app"]:
    U.safe(iam.delete_role_policy, RoleName=role, PolicyName="least-privilege")
    U.safe(iam.delete_role, RoleName=role)

Delete the AWS Config rules (if they were created).

In [ ]:
for rule in ["safedrive-sagemaker-notebook-no-internet", "safedrive-sagemaker-endpoint-config-kms"]:
    U.safe(config.delete_config_rule, ConfigRuleName=rule)

Delete the ECR repository.

In [ ]:
U.safe(ecr.delete_repository, repositoryName="safedrive-inference", force=True)

Delete the demo secret (without the recovery window).

In [ ]:
U.safe(U.client("secretsmanager").delete_secret, SecretId="safedrive/demo/vehicle-data-api-key",
       ForceDeleteWithoutRecovery=True)